# Benchmark Task Suite
A collection of 20 structured benchmark tasks for evaluating AI systems on real-world data analysis and economics problems.  
Each task has a defined input, expected output criteria, and an automated Python scoring function.

**Author:** Abigael Cherotich  
**Purpose:** AI Evaluation Engineering Portfolio — Project 3  
**Skills demonstrated:** Benchmark task design, scoring rubric development, evaluation harness structure, economics domain expertise

## What is a benchmark task?

A benchmark task is a structured test used to evaluate whether an AI system can perform a specific analytical task correctly.

Each task in this suite has four components — the same structure used by professional evaluation frameworks like OpenAI Evals, HELM, and MLE Bench:

| Component | Description |
|-----------|-------------|
| **Input** | The data and question given to the AI system |
| **Expected output** | What a correct answer looks like |
| **Scoring function** | Python code that checks if the output is correct |
| **Difficulty** | Easy / Medium / Hard |

This suite focuses on **data analysis and economics tasks** — domain areas where subject matter expertise matters.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ── Scoring utilities ──────────────────────────────────────────────────

def score_task(task_id, model_output, expected_output, scoring_fn):
    """
    Master scoring wrapper.
    Returns a standardised result dict for every task.
    """
    try:
        score, feedback = scoring_fn(model_output, expected_output)
        return {
            'task_id'      : task_id,
            'score'        : score,          # 0.0 to 1.0
            'passed'       : score >= 0.7,
            'feedback'     : feedback,
            'model_output' : str(model_output)[:200],
            'timestamp'    : datetime.now().strftime('%H:%M:%S')
        }
    except Exception as e:
        return {
            'task_id'  : task_id,
            'score'    : 0.0,
            'passed'   : False,
            'feedback' : f'Scoring error: {str(e)}',
            'error'    : True
        }

def numeric_score(model_val, expected_val, tolerance=0.05):
    """Score a numeric answer — full credit within tolerance, partial beyond."""
    if expected_val == 0:
        return 1.0 if model_val == 0 else 0.0
    pct_error = abs(model_val - expected_val) / abs(expected_val)
    if pct_error <= tolerance:
        return 1.0
    elif pct_error <= 0.15:
        return 0.7
    elif pct_error <= 0.30:
        return 0.4
    else:
        return 0.0

print("Scoring utilities loaded.")
print("Each task scores 0.0 to 1.0. Pass threshold = 0.7")

## Category 1 — Descriptive Statistics (Tasks 1–5)
These tasks test whether an AI can correctly compute and interpret basic statistical summaries from a dataset.

In [ ]:
# ── Shared dataset for Category 1 ─────────────────────────────────────
np.random.seed(42)
sales_df = pd.DataFrame({
    'region'  : np.random.choice(['North','South','East','West'], 120),
    'product' : np.random.choice(['A','B','C'], 120),
    'revenue' : np.random.normal(50000, 12000, 120).round(2),
    'units'   : np.random.randint(10, 200, 120),
    'quarter' : np.random.choice(['Q1','Q2','Q3','Q4'], 120)
})
# Inject a few nulls and an outlier
sales_df.loc[5, 'revenue']  = None
sales_df.loc[10, 'revenue'] = None
sales_df.loc[50, 'revenue'] = 250000.0  # outlier

print("Sales dataset loaded.")
print(sales_df.describe().round(2))

In [ ]:
# TASK 1 — Mean revenue by region
# ─────────────────────────────────────────────────────────────────────
task_1 = {
    'task_id'    : 'STAT-001',
    'difficulty' : 'Easy',
    'question'   : (
        'Given the sales_df dataset, compute the mean revenue '
        'for each region. Return a dict with region names as keys '
        'and mean revenue (rounded to 2 decimal places) as values. '
        'Exclude null values from the calculation.'
    ),
    'hint'       : 'Use groupby with mean(), dropna is handled automatically.'
}

# Correct expected output
expected_1 = sales_df.groupby('region')['revenue'].mean().round(2).to_dict()

# Scoring function
def score_task_1(model_output, expected):
    if not isinstance(model_output, dict):
        return 0.0, 'Output must be a dictionary'
    if set(model_output.keys()) != set(expected.keys()):
        return 0.3, f'Wrong regions: got {set(model_output.keys())}'
    scores = [numeric_score(model_output.get(r, 0), v)
              for r, v in expected.items()]
    avg_score = np.mean(scores)
    feedback = (f'Mean scores per region: '
                f'{[round(s,2) for s in scores]}. '
                f'Expected: {expected}')
    return round(avg_score, 2), feedback

# Simulate a model answer (correct)
model_answer_1 = expected_1.copy()
result_1 = score_task(task_1['task_id'], model_answer_1,
                       expected_1, score_task_1)

print(f"Task {task_1['task_id']} — {task_1['question'][:60]}...")
print(f"Expected output : {expected_1}")
print(f"Score           : {result_1['score']} | Passed: {result_1['passed']}")
print(f"Feedback        : {result_1['feedback']}")

In [ ]:
# TASK 2 — Identify the top revenue quarter
# ─────────────────────────────────────────────────────────────────────
task_2 = {
    'task_id'    : 'STAT-002',
    'difficulty' : 'Easy',
    'question'   : (
        'Which quarter had the highest total revenue across all regions? '
        'Return a single string: the quarter name (Q1, Q2, Q3, or Q4).'
    )
}

expected_2 = sales_df.groupby('quarter')['revenue'].sum().idxmax()

def score_task_2(model_output, expected):
    if str(model_output).strip().upper() == str(expected).strip().upper():
        return 1.0, f'Correct. Top quarter is {expected}'
    return 0.0, f'Wrong. Expected {expected}, got {model_output}'

model_answer_2 = expected_2
result_2 = score_task(task_2['task_id'], model_answer_2,
                       expected_2, score_task_2)

print(f"Task {task_2['task_id']} — Top revenue quarter")
print(f"Expected : {expected_2}")
print(f"Score    : {result_2['score']} | Passed: {result_2['passed']}")

# TASK 3 — Count products sold above revenue threshold
# ─────────────────────────────────────────────────────────────────────
task_3 = {
    'task_id'    : 'STAT-003',
    'difficulty' : 'Easy',
    'question'   : (
        'How many records in sales_df have revenue above $60,000? '
        'Return an integer. Exclude nulls.'
    )
}

expected_3 = int(sales_df[sales_df['revenue'] > 60000]['revenue'].count())

def score_task_3(model_output, expected):
    try:
        val = int(model_output)
        if val == expected:
            return 1.0, f'Correct: {expected} records above $60,000'
        return 0.0, f'Expected {expected}, got {val}'
    except:
        return 0.0, 'Output must be an integer'

result_3 = score_task(task_3['task_id'], expected_3,
                       expected_3, score_task_3)
print(f"\nTask {task_3['task_id']} — Records above $60k")
print(f"Expected : {expected_3}")
print(f"Score    : {result_3['score']} | Passed: {result_3['passed']}")

In [ ]:
# TASK 4 — Detect the revenue outlier
# ─────────────────────────────────────────────────────────────────────
task_4 = {
    'task_id'    : 'STAT-004',
    'difficulty' : 'Medium',
    'question'   : (
        'Using the z-score method with threshold 3.0, identify the '
        'index positions of outlier records in the revenue column. '
        'Return a list of integer index positions.'
    )
}

rev_clean  = sales_df['revenue'].dropna()
z_scores   = abs((rev_clean - rev_clean.mean()) / rev_clean.std())
expected_4 = sorted(rev_clean[z_scores > 3.0].index.tolist())

def score_task_4(model_output, expected):
    try:
        output_set   = set(int(x) for x in model_output)
        expected_set = set(expected)
        if output_set == expected_set:
            return 1.0, f'Correct outlier indices: {expected}'
        overlap = output_set & expected_set
        if overlap:
            return 0.5, (f'Partial: found {overlap}, '
                         f'missed {expected_set - output_set}')
        return 0.0, f'No correct outliers found. Expected {expected}'
    except:
        return 0.0, 'Output must be a list of integers'

result_4 = score_task(task_4['task_id'], expected_4,
                       expected_4, score_task_4)
print(f"Task {task_4['task_id']} — Outlier detection")
print(f"Expected indices : {expected_4}")
print(f"Score            : {result_4['score']} | Passed: {result_4['passed']}")

# TASK 5 — Revenue correlation with units
# ─────────────────────────────────────────────────────────────────────
task_5 = {
    'task_id'    : 'STAT-005',
    'difficulty' : 'Medium',
    'question'   : (
        'Compute the Pearson correlation between revenue and units sold. '
        'Round to 4 decimal places. Return a float.'
    )
}

expected_5 = round(float(sales_df[['revenue','units']].dropna()
                         .corr().loc['revenue','units']), 4)

def score_task_5(model_output, expected):
    s = numeric_score(float(model_output), expected, tolerance=0.02)
    return s, f'Expected {expected}, got {model_output}'

result_5 = score_task(task_5['task_id'], expected_5,
                       expected_5, score_task_5)
print(f"\nTask {task_5['task_id']} — Revenue-units correlation")
print(f"Expected : {expected_5}")
print(f"Score    : {result_5['score']} | Passed: {result_5['passed']}")

## Category 2 — Data Cleaning and Transformation (Tasks 6–10)

In [ ]:
# Shared messy dataset for Category 2
np.random.seed(7)
messy_df = pd.DataFrame({
    'id'      : range(1, 101),
    'country' : np.random.choice(['Kenya','Uganda','Tanzania',
                                   'KENYA','kenya','Unknown'], 100),
    'gdp_usd' : np.random.uniform(1e9, 5e11, 100),
    'year'    : np.random.choice([2019,2020,2021,2022,2023], 100),
    'category': np.random.choice(['A','B','C',None,'D'], 100)
})
messy_df.loc[10:14, 'gdp_usd'] = None
messy_df.loc[20, 'year']       = 9999   # invalid year

print("Messy economics dataset loaded.")
print(messy_df.head(8))
print(f"\nNulls:\n{messy_df.isnull().sum()}")

In [ ]:
# TASK 6 — Normalise country names
task_6 = {
    'task_id'    : 'CLEAN-001',
    'difficulty' : 'Easy',
    'question'   : (
        'Normalise the country column so all values are title-cased '
        '(first letter uppercase, rest lowercase). Return the unique '
        'normalised country values as a sorted list.'
    )
}

expected_6 = sorted(messy_df['country'].dropna()
                    .str.title().unique().tolist())

def score_task_6(model_output, expected):
    try:
        output_set   = set(str(v) for v in model_output)
        expected_set = set(expected)
        if output_set == expected_set:
            return 1.0, f'Correct: {sorted(expected_set)}'
        overlap = len(output_set & expected_set)
        partial = overlap / len(expected_set)
        return round(partial, 2), (f'{overlap}/{len(expected_set)} '
                                    f'correct values')
    except:
        return 0.0, 'Output must be a list'

result_6 = score_task(task_6['task_id'], expected_6,
                       expected_6, score_task_6)
print(f"Task {task_6['task_id']} — Normalise country names")
print(f"Expected : {expected_6}")
print(f"Score    : {result_6['score']} | Passed: {result_6['passed']}")

# TASK 7 — Flag invalid year values
task_7 = {
    'task_id'    : 'CLEAN-002',
    'difficulty' : 'Easy',
    'question'   : (
        'Identify records where the year is outside the valid range '
        '(2019–2023). Return a list of row index positions.'
    )
}

expected_7 = sorted(messy_df[~messy_df['year'].between(2019,2023)].index.tolist())

def score_task_7(model_output, expected):
    try:
        if sorted([int(x) for x in model_output]) == sorted(expected):
            return 1.0, f'Correct: invalid year at index {expected}'
        return 0.0, f'Expected {expected}, got {list(model_output)}'
    except:
        return 0.0, 'Output must be a list of integers'

result_7 = score_task(task_7['task_id'], expected_7,
                       expected_7, score_task_7)
print(f"\nTask {task_7['task_id']} — Flag invalid years")
print(f"Expected : {expected_7}")
print(f"Score    : {result_7['score']} | Passed: {result_7['passed']}")

In [ ]:
# TASK 8 — Impute missing GDP with median
task_8 = {
    'task_id'    : 'CLEAN-003',
    'difficulty' : 'Medium',
    'question'   : (
        'Fill null values in the gdp_usd column with the column median. '
        'Return the count of rows that were imputed (integer).'
    )
}

null_count_before = int(messy_df['gdp_usd'].isnull().sum())
expected_8        = null_count_before

def score_task_8(model_output, expected):
    try:
        if int(model_output) == expected:
            return 1.0, f'Correct: {expected} rows imputed'
        return 0.0, f'Expected {expected}, got {model_output}'
    except:
        return 0.0, 'Output must be an integer'

result_8 = score_task(task_8['task_id'], expected_8,
                       expected_8, score_task_8)
print(f"Task {task_8['task_id']} — GDP imputation count")
print(f"Expected : {expected_8} rows imputed")
print(f"Score    : {result_8['score']} | Passed: {result_8['passed']}")

# TASK 9 — Pivot GDP by country and year
task_9 = {
    'task_id'    : 'CLEAN-004',
    'difficulty' : 'Medium',
    'question'   : (
        'Create a pivot table showing mean GDP by country (rows) and '
        'year (columns). How many unique countries appear in the result? '
        'Return an integer.'
    )
}

pivot      = messy_df.pivot_table(values='gdp_usd',
                                   index='country',
                                   columns='year',
                                   aggfunc='mean')
expected_9 = len(pivot)

def score_task_9(model_output, expected):
    try:
        if int(model_output) == expected:
            return 1.0, f'Correct: {expected} unique countries in pivot'
        return 0.0, f'Expected {expected}, got {model_output}'
    except:
        return 0.0, 'Output must be an integer'

result_9 = score_task(task_9['task_id'], expected_9,
                       expected_9, score_task_9)
print(f"\nTask {task_9['task_id']} — Pivot table unique countries")
print(f"Expected : {expected_9}")
print(f"Score    : {result_9['score']} | Passed: {result_9['passed']}")

# TASK 10 — Category distribution after removing unknowns
task_10 = {
    'task_id'    : 'CLEAN-005',
    'difficulty' : 'Medium',
    'question'   : (
        'Remove rows where country is "Unknown" or category is null. '
        'What percentage of the original dataset remains? '
        'Return a float rounded to 2 decimal places.'
    )
}

cleaned    = messy_df[(messy_df['country'] != 'Unknown') &
                       (messy_df['category'].notna())]
expected_10 = round(len(cleaned) / len(messy_df) * 100, 2)

def score_task_10(model_output, expected):
    s = numeric_score(float(model_output), expected, tolerance=0.02)
    return s, f'Expected {expected}%, got {model_output}%'

result_10 = score_task(task_10['task_id'], expected_10,
                        expected_10, score_task_10)
print(f"\nTask {task_10['task_id']} — Percentage remaining after cleaning")
print(f"Expected : {expected_10}%")
print(f"Score    : {result_10['score']} | Passed: {result_10['passed']}")

## Category 3 — Economics & Research Analysis (Tasks 11–15)

In [ ]:
# Economic indicators dataset
np.random.seed(99)
countries = ['Kenya','Uganda','Tanzania','Rwanda','Ethiopia',
             'Nigeria','Ghana','Senegal','Egypt','Morocco']
years     = list(range(2015, 2024))

econ_data = []
for country in countries:
    base_gdp    = np.random.uniform(1e10, 5e11)
    base_inf    = np.random.uniform(2.0, 15.0)
    for year in years:
        econ_data.append({
            'country'       : country,
            'year'          : year,
            'gdp_usd'       : base_gdp * (1 + np.random.uniform(0.02, 0.08)),
            'inflation_pct' : base_inf + np.random.uniform(-2, 3),
            'unemployment'  : np.random.uniform(3.0, 20.0),
            'population_m'  : np.random.uniform(5, 220)
        })

econ_df = pd.DataFrame(econ_data)
print(f"Economic dataset: {econ_df.shape}")
print(econ_df.head(6).round(2))

In [ ]:
# TASK 11 — Highest average GDP country
task_11 = {
    'task_id'    : 'ECON-001',
    'difficulty' : 'Easy',
    'question'   : (
        'Which country had the highest average GDP (gdp_usd) '
        'across all years? Return the country name as a string.'
    )
}

expected_11 = econ_df.groupby('country')['gdp_usd'].mean().idxmax()

def score_task_11(model_output, expected):
    if str(model_output).strip().title() == str(expected).strip().title():
        return 1.0, f'Correct: {expected}'
    return 0.0, f'Expected {expected}, got {model_output}'

result_11 = score_task(task_11['task_id'], expected_11,
                        expected_11, score_task_11)
print(f"Task {task_11['task_id']} — Highest avg GDP country: {expected_11}")
print(f"Score: {result_11['score']} | Passed: {result_11['passed']}")

# TASK 12 — Year with lowest average inflation
task_12 = {
    'task_id'    : 'ECON-002',
    'difficulty' : 'Easy',
    'question'   : (
        'Which year had the lowest average inflation rate '
        'across all countries? Return an integer year.'
    )
}

expected_12 = int(econ_df.groupby('year')['inflation_pct'].mean().idxmin())

def score_task_12(model_output, expected):
    if int(model_output) == expected:
        return 1.0, f'Correct: {expected}'
    return 0.0, f'Expected {expected}, got {model_output}'

result_12 = score_task(task_12['task_id'], expected_12,
                        expected_12, score_task_12)
print(f"\nTask {task_12['task_id']} — Lowest avg inflation year: {expected_12}")
print(f"Score: {result_12['score']} | Passed: {result_12['passed']}")

In [ ]:
# TASK 13 — GDP per capita calculation
task_13 = {
    'task_id'    : 'ECON-003',
    'difficulty' : 'Medium',
    'question'   : (
        'Add a gdp_per_capita column (gdp_usd / population_m / 1e6). '
        'Which country had the highest mean GDP per capita? '
        'Return the country name.'
    )
}

econ_df['gdp_per_capita'] = econ_df['gdp_usd'] / (econ_df['population_m'] * 1e6)
expected_13 = econ_df.groupby('country')['gdp_per_capita'].mean().idxmax()

def score_task_13(model_output, expected):
    if str(model_output).strip().title() == str(expected).strip().title():
        return 1.0, f'Correct: {expected}'
    return 0.0, f'Expected {expected}, got {model_output}'

result_13 = score_task(task_13['task_id'], expected_13,
                        expected_13, score_task_13)
print(f"Task {task_13['task_id']} — Highest mean GDP per capita: {expected_13}")
print(f"Score: {result_13['score']} | Passed: {result_13['passed']}")

# TASK 14 — Inflation-unemployment correlation
task_14 = {
    'task_id'    : 'ECON-004',
    'difficulty' : 'Medium',
    'question'   : (
        'Compute the Pearson correlation between inflation_pct and '
        'unemployment. Round to 4 decimal places. Return a float. '
        '(This tests the Phillips Curve relationship in the data.)'
    )
}

expected_14 = round(float(econ_df[['inflation_pct','unemployment']]
                          .corr().loc['inflation_pct','unemployment']), 4)

def score_task_14(model_output, expected):
    s = numeric_score(float(model_output), expected, tolerance=0.02)
    return s, f'Expected {expected}, got {model_output}'

result_14 = score_task(task_14['task_id'], expected_14,
                        expected_14, score_task_14)
print(f"\nTask {task_14['task_id']} — Inflation-unemployment correlation: {expected_14}")
print(f"Score: {result_14['score']} | Passed: {result_14['passed']}")

# TASK 15 — Countries with consistently high unemployment
task_15 = {
    'task_id'    : 'ECON-005',
    'difficulty' : 'Hard',
    'question'   : (
        'Identify countries where unemployment was above 10% in more '
        'than 50% of the years in the dataset. Return a sorted list '
        'of country names.'
    )
}

pct_above = (econ_df.groupby('country')
             .apply(lambda g: (g['unemployment'] > 10).mean()))
expected_15 = sorted(pct_above[pct_above > 0.5].index.tolist())

def score_task_15(model_output, expected):
    try:
        output_set   = set(str(x).strip() for x in model_output)
        expected_set = set(expected)
        if output_set == expected_set:
            return 1.0, f'Correct: {sorted(expected_set)}'
        overlap = len(output_set & expected_set)
        score   = overlap / max(len(expected_set), 1)
        return round(score * 0.7, 2), (f'{overlap}/{len(expected_set)} '
                                        f'correct countries')
    except:
        return 0.0, 'Output must be a list of country names'

result_15 = score_task(task_15['task_id'], expected_15,
                        expected_15, score_task_15)
print(f"\nTask {task_15['task_id']} — High unemployment countries: {expected_15}")
print(f"Score: {result_15['score']} | Passed: {result_15['passed']}")

## Category 4 — AI Output Evaluation (Tasks 16–20)
These tasks simulate evaluating AI-generated analytical outputs — the core skill for MLE Bench roles.

In [ ]:
# Simulated AI model outputs for evaluation
ai_outputs = {
    'output_001': {
        'question' : 'What is the mean of [10, 20, 30, 40, 50]?',
        'ai_answer': '30',
        'correct'  : '30'
    },
    'output_002': {
        'question' : 'Is the correlation between X and Y (0.85) strong?',
        'ai_answer': 'Yes, 0.85 is a strong positive correlation.',
        'correct'  : 'strong positive'
    },
    'output_003': {
        'question' : 'What does an R² of 0.12 mean?',
        'ai_answer': ('An R² of 0.12 means the model explains 12% of '
                      'the variance in the target variable. '
                      'This is generally considered weak predictive power.'),
        'correct'  : 'explains 12% of variance, weak'
    },
    'output_004': {
        'question' : 'What is 15% of 2000?',
        'ai_answer': '250',      # deliberately wrong
        'correct'  : '300'
    },
    'output_005': {
        'question' : ('Given precision=0.95 and recall=0.60, '
                      'should this medical model be deployed?'),
        'ai_answer': ('With precision of 0.95 and recall of 0.60, '
                      'the model misses 40% of real positive cases. '
                      'For a medical application this recall is too low '
                      'and the model should not be deployed without '
                      'threshold adjustment.'),
        'correct'  : 'no deploy, recall too low'
    }
}

print("AI output evaluation tasks loaded.")
print(f"Number of outputs to evaluate: {len(ai_outputs)}")

In [ ]:
# TASK 16 — Score a numeric AI answer
task_16 = {
    'task_id'    : 'EVAL-001',
    'difficulty' : 'Easy',
    'question'   : (
        'Evaluate output_001: does the AI answer match the expected '
        'numeric answer? Return 1.0 for correct, 0.0 for wrong.'
    )
}

def score_task_16(model_output, expected):
    try:
        score = float(model_output)
        if 0.9 <= score <= 1.0:
            return 1.0, 'Correctly identified answer as right'
        elif score == 0.0:
            return 0.0, 'Incorrectly flagged correct answer as wrong'
        return 0.5, f'Partial score given: {score}'
    except:
        return 0.0, 'Output must be a float'

result_16 = score_task(task_16['task_id'], 1.0, 1.0, score_task_16)
print(f"Task {task_16['task_id']} — Score numeric AI answer")
print(f"AI answer: {ai_outputs['output_001']['ai_answer']} "
      f"(correct: {ai_outputs['output_001']['correct']})")
print(f"Score: {result_16['score']} | Passed: {result_16['passed']}")

# TASK 17 — Detect wrong numeric answer
task_17 = {
    'task_id'    : 'EVAL-002',
    'difficulty' : 'Easy',
    'question'   : (
        'Evaluate output_004: the AI answered 250 for "15% of 2000". '
        'Is the answer correct? Return 1.0 if correct, 0.0 if wrong.'
    )
}

result_17 = score_task(task_17['task_id'], 0.0, 0.0,
                        lambda o, e: (1.0, 'Correctly identified wrong answer')
                        if float(o) == 0.0
                        else (0.0, 'Failed to catch the error'))
print(f"\nTask {task_17['task_id']} — Detect wrong numeric answer")
print(f"AI answer: {ai_outputs['output_004']['ai_answer']} "
      f"(correct: {ai_outputs['output_004']['correct']})")
print(f"Score: {result_17['score']} | Passed: {result_17['passed']}")

In [ ]:
# TASK 18 — Evaluate reasoning quality
task_18 = {
    'task_id'    : 'EVAL-003',
    'difficulty' : 'Medium',
    'question'   : (
        'Evaluate output_005 (medical model deployment decision). '
        'Score on a 0–3 rubric: '
        '0 = wrong recommendation, '
        '1 = correct recommendation but no reasoning, '
        '2 = correct recommendation with partial reasoning, '
        '3 = correct recommendation with full reasoning including '
        'specific metrics and clinical context. '
        'Return an integer 0–3.'
    )
}

# output_005 gives correct recommendation with full metrics + clinical context
expected_18 = 3

def score_task_18(model_output, expected):
    try:
        val = int(model_output)
        if val == expected:
            return 1.0, f'Correct rubric score: {expected}'
        elif abs(val - expected) == 1:
            return 0.7, f'Off by 1: expected {expected}, got {val}'
        return 0.0, f'Expected {expected}, got {val}'
    except:
        return 0.0, 'Output must be integer 0–3'

result_18 = score_task(task_18['task_id'], expected_18,
                        expected_18, score_task_18)
print(f"Task {task_18['task_id']} — Reasoning quality rubric")
print(f"AI answer: {ai_outputs['output_005']['ai_answer'][:80]}...")
print(f"Expected rubric score: {expected_18}/3")
print(f"Score: {result_18['score']} | Passed: {result_18['passed']}")

# TASK 19 — Identify failure mode
task_19 = {
    'task_id'    : 'EVAL-004',
    'difficulty' : 'Medium',
    'question'   : (
        'output_004 contains an error. Classify the failure mode: '
        '"calculation_error", "hallucination", "incomplete_answer", '
        'or "correct". Return the failure mode string.'
    )
}

expected_19 = 'calculation_error'

def score_task_19(model_output, expected):
    if str(model_output).strip().lower() == expected.lower():
        return 1.0, f'Correct failure mode: {expected}'
    valid = ['calculation_error','hallucination',
             'incomplete_answer','correct']
    if str(model_output).strip().lower() in valid:
        return 0.3, f'Wrong category. Expected {expected}'
    return 0.0, f'Invalid category. Choose from {valid}'

result_19 = score_task(task_19['task_id'], expected_19,
                        expected_19, score_task_19)
print(f"\nTask {task_19['task_id']} — Failure mode classification")
print(f"Expected failure mode: {expected_19}")
print(f"Score: {result_19['score']} | Passed: {result_19['passed']}")

# TASK 20 — Batch evaluation summary
task_20 = {
    'task_id'    : 'EVAL-005',
    'difficulty' : 'Hard',
    'question'   : (
        'Evaluate all 5 AI outputs (output_001 to output_005). '
        'output_004 is wrong. All others are correct. '
        'What is the overall pass rate as a percentage? '
        'Return a float rounded to 1 decimal place.'
    )
}

expected_20 = round(4/5 * 100, 1)

def score_task_20(model_output, expected):
    s = numeric_score(float(model_output), expected, tolerance=0.01)
    return s, f'Expected {expected}%, got {model_output}%'

result_20 = score_task(task_20['task_id'], expected_20,
                        expected_20, score_task_20)
print(f"\nTask {task_20['task_id']} — Batch evaluation pass rate: {expected_20}%")
print(f"Score: {result_20['score']} | Passed: {result_20['passed']}")

## Full Benchmark Results

In [ ]:
import matplotlib.pyplot as plt

all_results = [result_1, result_2, result_3, result_4, result_5,
               result_6, result_7, result_8, result_9, result_10,
               result_11, result_12, result_13, result_14, result_15,
               result_16, result_17, result_18, result_19, result_20]

all_tasks   = [task_1, task_2, task_3, task_4, task_5,
               task_6, task_7, task_8, task_9, task_10,
               task_11, task_12, task_13, task_14, task_15,
               task_16, task_17, task_18, task_19, task_20]

total     = len(all_results)
passed    = sum(1 for r in all_results if r['passed'])
avg_score = np.mean([r['score'] for r in all_results])

print("=" * 60)
print("          BENCHMARK SUITE — FULL RESULTS")
print("=" * 60)
print(f"{'Task ID':<12} {'Category':<12} {'Difficulty':<10} "
      f"{'Score':>7} {'Pass':>6}")
print("-" * 60)

categories = (['Statistics']*5 + ['Cleaning']*5 +
              ['Economics']*5 + ['AI Eval']*5)

for task, result, cat in zip(all_tasks, all_results, categories):
    diff = task.get('difficulty','Med')
    icon = '✓' if result['passed'] else '✗'
    print(f"  {task['task_id']:<10} {cat:<12} {diff:<10} "
          f"{result['score']:>7.2f}  {icon}")

print("=" * 60)
print(f"  Total tasks    : {total}")
print(f"  Passed (>=0.7) : {passed}")
print(f"  Pass rate      : {passed/total*100:.1f}%")
print(f"  Average score  : {avg_score:.4f}")
print("=" * 60)

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scores = [r['score'] for r in all_results]
colors = ['mediumseagreen' if r['passed'] else 'tomato'
          for r in all_results]
task_ids = [t['task_id'] for t in all_tasks]

axes[0].bar(task_ids, scores, color=colors, alpha=0.8)
axes[0].axhline(y=0.7, color='gray', linestyle='--',
                lw=1.5, label='Pass threshold (0.7)')
axes[0].set_title('Score per Task', fontsize=13)
axes[0].set_ylabel('Score (0–1)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=9)

cat_scores = {}
for cat, result in zip(categories, all_results):
    cat_scores.setdefault(cat, []).append(result['score'])
cat_means = {c: np.mean(v) for c, v in cat_scores.items()}

axes[1].bar(cat_means.keys(), cat_means.values(),
            color=['steelblue','darkorange','mediumseagreen','mediumpurple'],
            alpha=0.8)
axes[1].axhline(y=0.7, color='gray', linestyle='--', lw=1.5)
axes[1].set_title('Average Score by Category', fontsize=13)
axes[1].set_ylabel('Average Score')
axes[1].set_ylim(0, 1.1)

plt.suptitle('Benchmark Task Suite — Evaluation Results', fontsize=14)
plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation — Benchmark Results

#### What the benchmark suite demonstrates

This suite tests an AI system across four skill categories that directly
mirror what MLE Bench and SWE Bench evaluate:

- **Statistics** (Tasks 1–5): Can the system compute correct summaries?
- **Data Cleaning** (Tasks 6–10): Can it handle messy real-world data?
- **Economics** (Tasks 11–15): Does it apply domain knowledge correctly?
- **AI Output Evaluation** (Tasks 16–20): Can it evaluate other AI outputs?

#### How the scoring functions work

Each scoring function returns a value between 0.0 and 1.0:
- **1.0** = fully correct
- **0.7–0.9** = partially correct — right direction, minor error
- **0.4–0.6** = partial credit — some correct elements
- **0.0–0.3** = incorrect

The 0.7 pass threshold mirrors professional evaluation frameworks —
partial credit is awarded for answers that show correct reasoning
even if the final value has minor errors.

#### What makes a good benchmark task

1. **Unambiguous expected output** — there is one correct answer
2. **Automated scoring** — no human needed to judge correctness
3. **Calibrated difficulty** — Easy/Medium/Hard tasks in each category
4. **Domain grounding** — tasks reflect real analytical work
5. **Failure mode coverage** — tasks that catch different error types

These five properties are what distinguish a professional benchmark
from a simple quiz — and they are what Turing's MLE Bench is built on.

**Author:** Abigael Cherotich — Data Analyst & AI Evaluation Specialist